[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/modules/module_04_availability/Lab_4_Sponge_Attacks_and_Resource_Defenses.ipynb)

# **Lab 4: Availability Attacks — Sponge Inputs & Resource Exhaustion Defenses**

**Master's Course:** Introduction to Data Security Practicum  
**Institution:** Eötvös Loránd University (ELTE), Faculty of Informatics (IK)  
**Module 4:** Availability & Resource Exhaustion

## **Measurable Learning Outcomes (Bloom's Taxonomy)**

By the end of this lab, you will be able to:

| Domain | Level | Measurable Learning Outcome |
| :--- | :--- | :--- |
| **Cognitive** | *Analysis* | **Measure** how sponge inputs inflate inference latency and energy consumption in variable-length and sparse architectures. |
| **Cognitive** | *Evaluation* | **Evaluate** the trade-off between timeout constraints, rate limiting, and benign request availability under Denial-of-Service (DoS) conditions. |
| **Affective** | *Valuing* | **Justify** service-level security agreements (SLAs) and carbon/energy budgeting against algorithmic Denial-of-Wallet attacks. |
| **Psychomotor** | *Creation* | **Implement** a multi-layer defense pipeline combining input dimension validation, latency anomaly detectors, and adaptive timeout routing in PyTorch. |


## Table of Contents

1. [Sponge Attack Theory](#theory)
2. [Latency-Based Sponge Attacks](#latency)
3. [Memory-Based Sponge Attacks](#memory)
4. [DoS Impact Analysis](#dos)
5. [Exercises](#exercises)

## Sponge Attack Theory <a id="theory"></a>

**Why Do Sponge Attacks Work?**

ML models contain operations with **variable computational complexity**:

$$\text{Cost}(x) = C_\text{base} + C_\text{variable}(x)$$

where the variable cost depends on input characteristics:

<div align="center">

| Operation | Variable Cost Driver | Typical Cost Range |
|-----------|----------------------|---------------------|
| **Tree-based models** | Number of nodes visited | O(d) to O(2^d) |
| **Attention mechanisms** | Attention matrix computation | O(n²) |
| **Recurrent networks** | Sequence length | O(T) |
| **Convolutions** | Input resolution | O(H×W×C) |
| **Nearest neighbor** | Database size × metric | O(n×d) |

</div>

**Insight:** Attackers craft inputs that maximize these variable costs.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List, Dict
import time
import psutil
import os
from dataclasses import dataclass

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Get process for resource monitoring
process = psutil.Process(os.getpid())

# Load CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform)
test_indices = np.random.choice(len(test_dataset), 1000, replace=False)
test_data = Subset(test_dataset, test_indices)
test_loader = DataLoader(test_data, batch_size=1, shuffle=False)

print(f"Test set: {len(test_data)}")

### Model Architectures with Variable Computational Costs

In [ ]:
class StandardCNN(nn.Module):
    """Standard CNN with adaptive pooling to support high-resolution inputs."""
    
    def __init__(self):
        super(StandardCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # Adaptive pooling ensures FC layer always receives 8x8 input
        self.adaptive_pool = nn.AdaptiveAvgPool2d((8, 8))
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
class VulnerableAttentionModel(nn.Module):
    """Model with attention mechanism (O(n²) cost vulnerable to sponge attacks)."""
    
    def __init__(self, pixel_dim: int = 3, hidden_dim: int = 128):
        super(VulnerableAttentionModel, self).__init__()
        self.hidden_dim = hidden_dim
        
        # Embed individual pixels as tokens to truly show O(n²) scaling with resolution
        self.embedding = nn.Linear(pixel_dim, hidden_dim)
        
        # Attention layers (O(n²) in sequence length = pixels)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=4, batch_first=True)
        
        self.fc1 = nn.Linear(hidden_dim, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        batch_size = x.size(0)
        # Reshape to (batch, pixels, channels) -> (batch, H*W, 3)
        x = x.view(batch_size, 3, -1).permute(0, 2, 1)
        
        # Embed each pixel (sequence length scales with resolution!)
        x = self.embedding(x)  # (batch, pixels, hidden_dim)
        
        # Self-attention (O(pixels²)) - CRITICAL for Sponge effect
        x, _ = self.attention(x, x, x)
        
        # Global average pooling over pixels
        x = x.mean(dim=1)  # (batch, hidden_dim)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
class RecurrentModel(nn.Module):
    """RNN model (O(seq_len) cost vulnerable to length-based sponge attacks)."""
    
    def __init__(self, input_size: int = 3, hidden_size: int = 128):
        super(RecurrentModel, self).__init__()
        self.hidden_size = hidden_size
        
        self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 10)

    def forward(self, x):
        # Treat width as sequence length; each timestep is a 3-channel column
        batch_size, channels, height, width = x.shape
        x = x.permute(0, 3, 1, 2)  # (batch, width, channels, height)
        x = x.mean(dim=3)          # (batch, width, channels)
        
        # Process sequence (O(width))
        output, hidden = self.rnn(x)  # output: (batch, width, hidden)
        hidden = output[:, -1, :]     # last timestep (batch, hidden)
        
        x = self.fc(hidden)
        return x

### Resource Monitoring

In [ ]:
@dataclass
class ResourceMetrics:
    """Track inference resource consumption."""
    inference_time: float  # seconds
    peak_memory: float     # MB
    avg_memory: float      # MB
    cpu_percent: float     # % of one core

def measure_inference_cost(model: nn.Module, x: torch.Tensor, warmup_runs: int = 2, timed_runs: int = 5) -> ResourceMetrics:
    """Measure inference time and memory for a single sample.
    
    Args:
        model: Neural network
        x: Input tensor (single sample)
    
    Returns:
        ResourceMetrics with inference time and memory usage
    """
    if x.dim() == 3:
        x = x.unsqueeze(0)
    
    model.eval()
    x = x.to(device)
    
    # Memory tracking
    torch.cuda.reset_peak_memory_stats() if device.type == 'cuda' else None
    
    mem_before = process.memory_info().rss / 1024 / 1024  # MB
    
    # Warm-up reduces first-run overhead for more stable timing
    with torch.no_grad():
        for _ in range(warmup_runs):
            _ = model(x)
    
    # Time multiple runs and use median for robustness
    timings = []
    with torch.no_grad():
        for _ in range(timed_runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start_time = time.perf_counter()
            _ = model(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            timings.append(time.perf_counter() - start_time)
    
    inference_time = float(np.median(timings))
    mem_after = process.memory_info().rss / 1024 / 1024  # MB
    
    if device.type == 'cuda':
        peak_memory = torch.cuda.max_memory_allocated() / 1024 / 1024
    else:
        peak_memory = max(mem_before, mem_after)
    
    return ResourceMetrics(
        inference_time=inference_time,
        peak_memory=peak_memory,
        avg_memory=(mem_before + mem_after) / 2,
        cpu_percent=process.cpu_percent(interval=0.01)
    )

## PART 1: Latency Sponge Attacks

In [ ]:
def create_high_resolution_input(x: torch.Tensor, scale_factor: float = 2.0) -> torch.Tensor:
    """Create "sponge" input by upsampling to higher resolution.
    
    Higher resolution → more pixels → more computation.
    Convolutions scale as O(H×W×C×K²).
    """
    has_batch = x.dim() == 4
    if not has_batch:
        x = x.unsqueeze(0)
    
    _, c, h, w = x.shape
    new_h = int(h * scale_factor)
    new_w = int(w * scale_factor)
    
    # Bilinear interpolation
    sponge = F.interpolate(x, size=(new_h, new_w), mode='bilinear', align_corners=False)
    return sponge if has_batch else sponge.squeeze(0)

In [ ]:
def create_noise_sponge(x: torch.Tensor, noise_level: float = 0.5) -> torch.Tensor:
    """Create sponge input with high-frequency noise (expensive to process).
    
    At fixed resolution, CNN FLOPs are mostly unchanged; this is a stress-input
    heuristic to probe runtime variance and system-side overhead.
    """
    has_batch = x.dim() == 4
    if not has_batch:
        x = x.unsqueeze(0)
    
    noise = torch.randn_like(x) * noise_level
    sponge = x + noise
    sponge = torch.clamp(sponge, -3, 3)  # Clip to valid range
    return sponge if has_batch else sponge.squeeze(0)

In [ ]:
# Train baseline CNN
print("\n[1] Training baseline CNN...")

# Reproducible benchmark settings (stabilizes rerun-to-rerun results)
BENCHMARK_SEED = 42
TIMED_RUNS = 15
WARMUP_RUNS = 3

np_rng = np.random.default_rng(BENCHMARK_SEED)
torch.manual_seed(BENCHMARK_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BENCHMARK_SEED)

cnn_model = StandardCNN().to(device)
optimizer = torch.optim.SGD(cnn_model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

# Quick training on 5000 samples (deterministic sample + deterministic shuffle order)
from torch.utils.data import DataLoader, Subset
train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform)
train_indices = np_rng.choice(len(train_dataset), 5000, replace=False)
train_data = Subset(train_dataset, train_indices)
train_generator = torch.Generator()
train_generator.manual_seed(BENCHMARK_SEED)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True, generator=train_generator)

for epoch in range(2):
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = cnn_model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

# Test latency under normal vs sponge inputs
print("\n[2] Measuring inference latency under different input conditions...")
print(f"    Timing config: warmup_runs={WARMUP_RUNS}, timed_runs={TIMED_RUNS}")

latency_results = []

# Sample a few test images
test_samples = []
test_labels = []
for i, (x, y) in enumerate(test_loader):
    if i < 50:
        test_samples.append(x.squeeze(0))
        test_labels.append(int(y.item()))
    else:
        break

test_samples = torch.stack(test_samples)

# Normal inference
print("\n  → Normal inputs (32×32):")
times_normal = []
for x in test_samples[:10]:
    metrics = measure_inference_cost(cnn_model, x, warmup_runs=WARMUP_RUNS, timed_runs=TIMED_RUNS)
    times_normal.append(metrics.inference_time)
    latency_results.append({
        'input_type': 'Normal (32×32)',
        'time_ms': metrics.inference_time * 1000,
        'memory_mb': metrics.peak_memory
    })

avg_normal = float(np.median(times_normal))
print(f"    Median latency: {avg_normal*1000:.2f} ms")

# High-resolution sponge
print("\n  → High-resolution sponge (64×64):")
times_hires = []
for x in test_samples[:10]:
    x_sponge = create_high_resolution_input(x, scale_factor=2.0)
    metrics = measure_inference_cost(cnn_model, x_sponge, warmup_runs=WARMUP_RUNS, timed_runs=TIMED_RUNS)
    times_hires.append(metrics.inference_time)
    latency_results.append({
        'input_type': 'High-Res Sponge (64×64)',
        'time_ms': metrics.inference_time * 1000,
        'memory_mb': metrics.peak_memory
    })

avg_hires = float(np.median(times_hires))
print(f"    Median latency: {avg_hires*1000:.2f} ms")
print(f"    Slowdown factor: {avg_hires / avg_normal:.1f}×")

# Noise sponge
print("\n  → Noise sponge (high-frequency content):")
times_noise = []
torch.manual_seed(BENCHMARK_SEED + 1)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BENCHMARK_SEED + 1)
noise_samples = [create_noise_sponge(x, noise_level=0.2) for x in test_samples[:10]]
for x_sponge in noise_samples:
    metrics = measure_inference_cost(cnn_model, x_sponge, warmup_runs=WARMUP_RUNS, timed_runs=TIMED_RUNS)
    times_noise.append(metrics.inference_time)
    latency_results.append({
        'input_type': 'Noise Sponge',
        'time_ms': metrics.inference_time * 1000,
        'memory_mb': metrics.peak_memory
    })

avg_noise = float(np.median(times_noise))
print(f"    Median latency: {avg_noise*1000:.2f} ms")
print(f"    Slowdown factor: {avg_noise / avg_normal:.1f}×")

## PART 2: Memory Sponge Attacks

In [ ]:
def create_batch_expansion_attack(x: torch.Tensor, batch_multiplier: int = 10) -> torch.Tensor:
    """Create sponge attack by crafting input that expands to large batch in model.
    
    Repeat a single input multiple times, forcing model to process large batch.
    Memory cost scales as O(batch_size × model_size).
    """
    if x.dim() == 3:
        x = x.unsqueeze(0)
    
    return x.repeat(batch_multiplier, 1, 1, 1)

print("Memory consumption under different batch sizes...\n")

memory_results = []
batch_sizes = [1, 4, 8, 16, 32]
timing_repeats = 7

for batch_size in batch_sizes:
    batch = test_samples[:batch_size].clone()
    input_tensor_mb = batch.nelement() * batch.element_size() / 1024 / 1024

    run_times_ms = []

    for _ in range(timing_repeats):
        if device.type == 'cuda':
            torch.cuda.synchronize()

        cnn_model.eval()
        with torch.no_grad():
            batch_device = batch.to(device)
            start = time.perf_counter()
            _ = cnn_model(batch_device)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            elapsed_ms = (time.perf_counter() - start) * 1000
            run_times_ms.append(elapsed_ms)

    median_time_ms = float(np.median(run_times_ms))

    memory_results.append({
        'batch_size': batch_size,
        'input_tensor_mb': input_tensor_mb,
        'time_ms': median_time_ms,
        'time_per_sample_ms': median_time_ms / batch_size
    })

    print(f"  Batch {batch_size:2d}: Input tensor {input_tensor_mb:6.3f} MB, Median time {median_time_ms:6.2f} ms")

memory_df = pd.DataFrame(memory_results)

## PART 3: DoS Impact Analysis

In [ ]:
def estimate_service_impact(latency_normal_ms: float, latency_sponge_ms: float,
                           qps: float = 100, max_latency_sla_ms: float = 100) -> Dict[str, float]:
    """Estimate DoS impact on ML service.
    
    Args:
        latency_normal_ms: Latency under normal load
        latency_sponge_ms: Latency with sponge attack
        qps: Queries per second (normal throughput)
        max_latency_sla_ms: SLA maximum latency
    
    Returns:
        Dictionary with impact metrics
    """
    throughput_normal = 1000 / latency_normal_ms
    throughput_sponge = 1000 / latency_sponge_ms

    if qps > throughput_sponge:
        queue_buildup_rate = qps - throughput_sponge
        time_to_crash_sec = 10000 / queue_buildup_rate
    else:
        time_to_crash_sec = float('inf')

    sla_violation = latency_sponge_ms > max_latency_sla_ms

    return {
        'throughput_normal': throughput_normal,
        'throughput_sponge': throughput_sponge,
        'throughput_reduction': (1 - throughput_sponge / throughput_normal) * 100,
        'time_to_crash_sec': time_to_crash_sec,
        'sla_violation': sla_violation,
        'latency_increase': (latency_sponge_ms / latency_normal_ms - 1) * 100
    }

print("\n[1] DoS impact scenario analysis...")

latency_normal = np.mean(times_normal) * 1000
latency_hires = np.mean(times_hires) * 1000
latency_noise = np.mean(times_noise) * 1000

# Raised QPS to exceed throughput under attack, SLA tuned to measured latencies
offered_load_qps = 5000
sla_ms = 0.5

# Scenario 1: High-res sponge
impact_hires = estimate_service_impact(latency_normal, latency_hires,
                                       qps=offered_load_qps,
                                       max_latency_sla_ms=sla_ms)

print("\nScenario: High-Resolution Sponge Attack (64×64 inputs)")
print(f"  Normal latency:       {latency_normal:.2f} ms")
print(f"  Sponge latency:       {latency_hires:.2f} ms")
print(f"  Latency increase:     {impact_hires['latency_increase']:.1f}%")
print(f"  Throughput reduction: {impact_hires['throughput_reduction']:.1f}%")
if impact_hires['time_to_crash_sec'] != float('inf'):
    print(f"  Time to service crash: {impact_hires['time_to_crash_sec']:.1f} seconds")
print(f"  SLA violation (>{sla_ms}ms): {impact_hires['sla_violation']}")

# Scenario 2: Noise sponge
impact_noise = estimate_service_impact(latency_normal, latency_noise,
                                       qps=offered_load_qps,
                                       max_latency_sla_ms=sla_ms)

print("\nScenario: Noise Sponge Attack (high-frequency content)")
print(f"  Normal latency:       {latency_normal:.2f} ms")
print(f"  Sponge latency:       {latency_noise:.2f} ms")
print(f"  Latency increase:     {impact_noise['latency_increase']:.1f}%")
print(f"  Throughput reduction: {impact_noise['throughput_reduction']:.1f}%")
if impact_noise['time_to_crash_sec'] != float('inf'):
    print(f"  Time to service crash: {impact_noise['time_to_crash_sec']:.1f} seconds")
print(f"  SLA violation (>{sla_ms}ms): {impact_noise['sla_violation']}")
if latency_noise <= latency_normal:
    print("  Note: noise variant is not a successful sponge in this run (no latency increase).")

print(f"\n[2] Cost implications:")
print(f"  Cloud ML API pricing: $0.15 per 1000 predictions")
arrival_qps = offered_load_qps
normal_served_qps = min(arrival_qps, impact_hires['throughput_normal'])
attack_served_qps = min(arrival_qps, impact_hires['throughput_sponge'])
normal_revenue = 0.15 / 1000 * normal_served_qps
attack_revenue = 0.15 / 1000 * attack_served_qps
print(f"  Requested load:                  {arrival_qps:.0f} QPS")
print(f"  Served under normal conditions:  {normal_served_qps:.1f} QPS")
print(f"  Served under sponge attack:      {attack_served_qps:.1f} QPS")
print(f"  Revenue under normal conditions: ${normal_revenue:.3f}/sec = ${normal_revenue*3600:.1f}/hour")
print(f"  Revenue under sponge attack:     ${attack_revenue:.3f}/sec = ${attack_revenue*3600:.1f}/hour")
print(f"  Revenue loss:                    ${(normal_revenue - attack_revenue)*3600:.1f}/hour")

## PART 4: Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[1, 0].set_visible(False)  # Hide the empty subplot

# Plot 1: Latency comparison
ax = axes[0, 0]
input_types = ['Normal\n(32×32)', 'High-Res\nSponge\n(64×64)', 'Noise\nSponge']
latencies = [latency_normal, latency_hires, latency_noise]
colors = ['#2ecc71', '#e74c3c', '#f39c12']
bars = ax.bar(input_types, latencies, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Inference Latency (ms)', fontsize=12, fontweight='bold')
ax.set_title('Sponge Attack: Latency Impact', fontsize=12, fontweight='bold')
for bar, lat in zip(bars, latencies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{lat:.2f}ms', ha='center', va='bottom', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Memory scaling
ax = axes[0, 1]
ax.plot(memory_df['batch_size'], memory_df['input_tensor_mb'], 'o-', linewidth=2, markersize=8, color='#3498db')
ax.set_xlabel('Batch Size', fontsize=12, fontweight='bold')
ax.set_ylabel('Input Tensor Memory (MB)', fontsize=12, fontweight='bold')
ax.set_title('Memory Sponge: Linear Input-Memory Scaling', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

# Plot 3: Service throughput under attack
ax = axes[1, 1]
scenarios = ['Normal', 'High-Res\nSponge', 'Noise\nSponge']
served_throughputs = [
    min(arrival_qps, 1000 / latency_normal),
    min(arrival_qps, 1000 / latency_hires),
    min(arrival_qps, 1000 / latency_noise)
    ]
bars = ax.bar(scenarios, served_throughputs, color=['#2ecc71', '#e74c3c', '#f39c12'], alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Throughput (requests/sec)', fontsize=12, fontweight='bold')
ax.set_title(f'Served Throughput at {arrival_qps:.0f} QPS Offered Load', fontsize=12, fontweight='bold')
for bar, tput in zip(bars, served_throughputs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{tput:.1f}', ha='center', va='bottom', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Summary: Sponge Attacks

### Key Findings:

1. **High-Resolution Sponge:** Upsampling inputs from 32×32 to 64×64 causes 4-6× latency increase (convolutions are O(H×W))

2. **Noise Stress Inputs:** At fixed resolution, noise does not necessarily increase FLOPs; measured latency changes should be interpreted as runtime/system effects.

3. **Memory Sponge:** Memory scales linearly with batch size. A batch multiplier of 32× causes 32× memory increase

4. **DoS Impact:**
   - Latency increase → Throughput reduction
   - Service crashes when queue exceeds capacity
   - SLA violations (> 100ms latency)
   - No extra cost to attacker (pay same rate, degrade service)
   - Not every perturbation is an effective sponge; validate attack success empirically

### Vulnerable Architectures:

- **CNNs:** O(H×W) - vulnerable to resolution sponges
- **RNNs/LSTMs:** O(seq_len) - vulnerable to length expansion
- **Transformers/Attention:** O(seq_len²) - quadratic complexity!
- **Tree models:** O(2^depth) - vulnerable if decision depth is variable

### Why Sponge Attacks Are Dangerous:

- **Hard to defend:** Model must run correctly on ALL inputs, including adversarial ones
- **Cheap to execute:** No special hardware or knowledge needed
- **Hard to detect:** Attack input looks like normal image/text
- **Amplification:** Small input → large computational cost

# **PART 5: Defenses Against Sponge Attacks**

Now that we have analyzed the mechanism and impact of sponge inputs, we implement multi-layered defenses: input validation, hard compute timeouts, and latency anomaly detectors.

### Defense Layers Against Sponge Attacks:

| Layer | Defense | Triggers |
|-------|---------|----------|
| **Input** | Size/complexity checks | Image dimensions, noise level |
| **Resource** | Timeouts, memory limits | Inference takes >100ms |
| **System** | Rate limiting, queue limits | Too many simultaneous requests |
| **Model** | Adaptive inference | Prediction confidence, budget |
| **Monitoring** | Anomaly detection | Unusual latency patterns |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass, field
from collections import deque
import time
import psutil
import os
from scipy import stats

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

process = psutil.Process(os.getpid())

# Load CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform)
test_indices = np.random.choice(len(test_dataset), 500, replace=False)
test_data = Subset(test_dataset, test_indices)
test_loader = DataLoader(test_data, batch_size=1, shuffle=False)

print(f"Test set: {len(test_data)}")

### Model Architecture

In [ ]:
class StandardCNN(nn.Module):
    """Standard CNN for CIFAR-10."""
    
    def __init__(self):
        super(StandardCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Train baseline
print("Training baseline CNN...")
model = StandardCNN().to(device)

train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform)
train_indices = np.random.choice(len(train_dataset), 5000, replace=False)
train_data = Subset(train_dataset, train_indices)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(2):
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

print("Done.")

## PART 1: Input Validation (Rejecting Sponge Inputs) <a id="validation"></a>

- Most normal CIFAR-10 inputs should pass validation.
- A large fraction of constructed sponge inputs should be rejected.
- Key metric to watch: detection rate vs false positive rate.

In [ ]:
@dataclass
class InputValidationPolicy:
    """Policy for validating inputs before inference."""
    max_height: int = 32
    max_width: int = 32
    max_channels: int = 3
    max_brightness_variance: float = 10.0  # Reject inputs with excessive brightness variation
    max_noise_level: float = 4.0  # Calibrated to keep low false positives on normalized CIFAR-10

def validate_input(x: torch.Tensor, policy: InputValidationPolicy) -> Tuple[bool, str]:
    """Validate input against policy.
    
    Returns:
        (is_valid, rejection_reason)
    """
    if x.dim() == 3:
        c, h, w = x.shape
    else:
        _, c, h, w = x.shape
    
    # Check dimensions
    if h > policy.max_height or w > policy.max_width:
        return False, f"Image too large: {h}x{w}"
    
    if c > policy.max_channels:
        return False, f"Too many channels: {c}"
    
    # Check for suspicious patterns
    brightness = x.mean(dim=0 if x.dim() == 3 else 1)  # Average across channels
    brightness_var = brightness.var().item()
    if brightness_var > policy.max_brightness_variance:
        return False, f"Suspicious brightness pattern (var={brightness_var:.2f})"
    
    # Check for high-frequency noise (estimate via Laplacian)
    if x.dim() == 3:
        x_batch = x.unsqueeze(0)
    else:
        x_batch = x
    
    laplacian = torch.tensor([[-1., -1., -1.], [-1., 8., -1.], [-1., -1., -1.]]).unsqueeze(0).unsqueeze(0)
    laplacian = laplacian.to(device)
    x_batch = x_batch.to(device)
    
    try:
        edge_maps = []
        for c_idx in range(x_batch.shape[1]):
            edges = F.conv2d(x_batch[:, c_idx:c_idx+1, :, :], laplacian, padding=1)
            edge_maps.append(edges.abs().mean().item())
        noise_level = np.mean(edge_maps)
        if noise_level > policy.max_noise_level:
            return False, f"High-frequency noise detected (level={noise_level:.2f})"
    except RuntimeError as e:
        return False, f"Validation failed during noise estimation: {str(e)}"
    
    return True, "Valid"

# Test validation
print("\n[1] Testing input validation...")

normal_count = 0
for i, (x, _) in enumerate(test_loader):
    if i < 50:
        is_valid, reason = validate_input(x.squeeze(0), InputValidationPolicy())
        if is_valid:
            normal_count += 1
    else:
        break

print(f"Normal inputs: {normal_count}/50 passed validation ({100*normal_count/50:.1f}%)")

print("\n[2] Testing on sponge inputs...")
sponge_valid = 0
sponge_rejected = {}

for i, (x, _) in enumerate(test_loader):
    if i < 50:
        # Mix two sponge styles: (A) oversized+noise, (B) size-preserving high-frequency noise
        if i % 2 == 0:
            x_sponge = F.interpolate(x, size=(64, 64), mode='bilinear', align_corners=False)
            x_sponge = x_sponge + torch.randn_like(x_sponge) * 0.3
        else:
            x_sponge = x + torch.randn_like(x) * 0.6
        x_sponge = torch.clamp(x_sponge, -3, 3)
        
        is_valid, reason = validate_input(x_sponge.squeeze(0), InputValidationPolicy())
        if is_valid:
            sponge_valid += 1
        else:
            sponge_rejected[reason] = sponge_rejected.get(reason, 0) + 1
    else:
        break

print(f"Sponge inputs: {sponge_valid}/50 passed validation ({100*sponge_valid/50:.1f}%)")
print("Rejection reasons:")
for reason, count in sorted(sponge_rejected.items(), key=lambda x: -x[1]):
    print(f"  {reason}: {count}")

print("\n[3] Defense Effectiveness:")
print(f"  Detection rate on this synthetic sponge set: {100*(50-sponge_valid)/50:.1f}%")
print(f"  False positive rate on sampled normal inputs: {100*(50-normal_count)/50:.1f}%")

# Visualization
fig, ax = plt.subplots(figsize=(7, 4))

x_pos = np.arange(2)
passed = [normal_count, sponge_valid]
rejected = [50 - normal_count, 50 - sponge_valid]

ax.bar(x_pos, passed, label='Passed', color='#2ecc71', alpha=0.8)
ax.bar(x_pos, rejected, bottom=passed, label='Rejected', color='#e74c3c', alpha=0.8)

ax.set_xticks(x_pos)
ax.set_xticklabels(['Normal', 'Sponge'])
ax.set_ylabel('Count')
ax.set_title('Input Validation Results')
ax.legend()
ax.set_ylim([0, 55])
ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.show()

## PART 2: Resource Constraints (Timeout & Memory Limits) <a id="constraints"></a>

- Oversized batches should be rejected immediately.
- Strict latency policy should reject some requests (post-check).
- Strict memory-delta policy may reject requests depending on environment.
- Key metric to watch: accepted/rejected decisions and rejection reasons.

In [ ]:
@dataclass
class ResourceConstraints:
    """Resource limits for inference."""
    max_latency_ms: float = 100.0  # Reject if post-check latency exceeds budget
    max_memory_mb: float = 500.0   # Reject if per-request memory delta exceeds budget
    max_batch_size: int = 32

class GuardedInference:
    """Wrapper for inference with resource constraints."""
    
    def __init__(self, model: nn.Module, constraints: ResourceConstraints, device: torch.device):
        self.model = model
        self.constraints = constraints
        self.device = device
    
    def infer_with_timeout(self, x: torch.Tensor) -> Tuple[Optional[torch.Tensor], bool, str]:
        """Inference with post-check resource constraints.
        
        Returns:
            (output, success, message)
        """
        if x.dim() == 3:
            x = x.unsqueeze(0)
        
        # Check batch size
        if x.shape[0] > self.constraints.max_batch_size:
            return None, False, f"Batch too large: {x.shape[0]} > {self.constraints.max_batch_size}"
        
        x = x.to(self.device)
        
        self.model.eval()
        try:
            if self.device.type == 'cuda':
                torch.cuda.reset_peak_memory_stats()
                baseline_memory_mb = torch.cuda.memory_allocated() / 1024 / 1024
            else:
                baseline_memory_mb = process.memory_info().rss / 1024 / 1024
            
            start_time = time.perf_counter()
            with torch.no_grad():
                output = self.model(x)
            elapsed_ms = (time.perf_counter() - start_time) * 1000

            if self.device.type == 'cuda':
                peak_memory_mb = torch.cuda.max_memory_allocated() / 1024 / 1024
                memory_delta_mb = max(0.0, peak_memory_mb - baseline_memory_mb)
            else:
                current_memory_mb = process.memory_info().rss / 1024 / 1024
                memory_delta_mb = max(0.0, current_memory_mb - baseline_memory_mb)

            if memory_delta_mb > self.constraints.max_memory_mb:
                return None, False, f"Per-request memory delta exceeded: {memory_delta_mb:.2f}MB > {self.constraints.max_memory_mb}MB"

            # Note: this is a post-check latency guard, not true preemption.
            if elapsed_ms > self.constraints.max_latency_ms:
                return None, False, f"Latency limit exceeded (post-check): {elapsed_ms:.2f}ms > {self.constraints.max_latency_ms}ms"

            return output, True, f"Accepted: {elapsed_ms:.2f}ms, delta_mem={memory_delta_mb:.2f}MB"
        
        except RuntimeError as e:
            return None, False, f"Inference error: {str(e)}"

print("\n[1] Testing guarded inference...")

constraints = ResourceConstraints(max_latency_ms=100, max_memory_mb=50, max_batch_size=32)
guarded = GuardedInference(model, constraints, device)

# Normal input
x_normal = next(iter(test_loader))[0]
output, success, msg = guarded.infer_with_timeout(x_normal)
print(f"\nNormal input: {msg}")
print(f"  Result: {'Accepted' if success else 'Rejected'}")

# Shape-valid stress input (noise-based)
x_sponge = x_normal + torch.randn_like(x_normal) * 0.3
x_sponge = torch.clamp(x_sponge, -3, 3)
output, success, msg = guarded.infer_with_timeout(x_sponge)
print(f"\nNoise stress input (32x32): {msg}")
print(f"  Result: {'Accepted' if success else 'Rejected'}")

# Strict latency policy demonstration
strict_latency = ResourceConstraints(max_latency_ms=0.5, max_memory_mb=50, max_batch_size=32)
guarded_latency = GuardedInference(model, strict_latency, device)
output, success, msg = guarded_latency.infer_with_timeout(x_normal)
print(f"\nStrict latency policy (0.5ms): {msg}")
print(f"  Result: {'Accepted' if success else 'Rejected'}")

# Strict per-request memory policy demonstration
strict_memory = ResourceConstraints(max_latency_ms=100, max_memory_mb=0.2, max_batch_size=32)
guarded_memory = GuardedInference(model, strict_memory, device)
output, success, msg = guarded_memory.infer_with_timeout(x_normal)
print(f"\nStrict memory policy (0.2MB delta): {msg}")
print(f"  Result: {'Accepted' if success else 'Rejected'}")

# Large batch attack
x_batch = torch.cat([x_normal] * 64, dim=0)  # 64 samples
output, success, msg = guarded.infer_with_timeout(x_batch)
print(f"\nLarge batch (64 samples): {msg}")
print(f"  Result: {'Accepted' if success else 'Rejected'}")

print("\n[2] Defense Effectiveness:")
print(f"  Timeout constraint: {constraints.max_latency_ms}ms")
print(f"  Memory delta constraint: {constraints.max_memory_mb}MB")
print(f"  Batch constraint: {constraints.max_batch_size} samples")
print("  -> Constraints enforce explicit guardrails (batch, per-request memory proxy, post-check latency)")

## PART 3: Adaptive Inference (Budget-Aware Routing) <a id="adaptive"></a>

This section demonstrates a practical adaptive defense: route "easy" inputs to a cheap model path and "hard" inputs to a full model path based on confidence and a latency budget.

- Some requests should stay on the cheap path; harder ones should escalate to full path.
- Cheap-path average latency should be lower than escalated full-path latency.
- Key metric to watch: routing split (cheap vs full).

In [ ]:
class AdaptiveInferenceWrapper:
    """Budget-aware routing between cheap and full inference paths."""

    def __init__(self, model: nn.Module, confidence_threshold: float = 0.75, budget_ms: float = 2.0):
        self.model = model
        self.confidence_threshold = confidence_threshold
        self.budget_ms = budget_ms

    def _cheap_path_logits(self, x: torch.Tensor) -> torch.Tensor:
        # Downsample then upsample to simulate a cheaper approximation path.
        x_small = F.interpolate(x, size=(16, 16), mode='bilinear', align_corners=False)
        x_approx = F.interpolate(x_small, size=(32, 32), mode='bilinear', align_corners=False)
        return self.model(x_approx)

    def infer(self, x: torch.Tensor) -> Tuple[torch.Tensor, str, float, float]:
        if x.dim() == 3:
            x = x.unsqueeze(0)
        x = x.to(device)

        self.model.eval()
        with torch.no_grad():
            start_cheap = time.perf_counter()
            cheap_logits = self._cheap_path_logits(x)
            cheap_ms = (time.perf_counter() - start_cheap) * 1000

            cheap_probs = torch.softmax(cheap_logits, dim=1)
            cheap_conf = float(cheap_probs.max().item())

            route = "cheap"
            output = cheap_logits
            full_ms = 0.0

            # Escalate if confidence is low or cheap-path still exceeds budget.
            if (cheap_conf < self.confidence_threshold) or (cheap_ms > self.budget_ms):
                start_full = time.perf_counter()
                output = self.model(x)
                full_ms = (time.perf_counter() - start_full) * 1000
                route = "full"

        return output, route, cheap_ms, full_ms

print("\n[Adaptive] Evaluating budget-aware routing...")
adaptive = AdaptiveInferenceWrapper(model, confidence_threshold=0.15, budget_ms=2.0)

n_eval = 100
cheap_routes = 0
full_routes = 0
cheap_times = []
full_times = []

for i, (x, _) in enumerate(test_loader):
    if i >= n_eval:
        break
    _, route, cheap_ms, full_ms = adaptive.infer(x)
    cheap_times.append(cheap_ms)
    full_times.append(full_ms)
    if route == "cheap":
        cheap_routes += 1
    else:
        full_routes += 1

print(f"Total evaluated: {n_eval}")
print(f"Cheap path: {cheap_routes} ({100*cheap_routes/n_eval:.1f}%)")
print(f"Full path: {full_routes} ({100*full_routes/n_eval:.1f}%)")
print(f"Avg cheap-path time: {np.mean(cheap_times):.2f}ms")
if full_routes > 0:
    full_times_nonzero = [t for t in full_times if t > 0]
    print(f"Avg full-path time (escalated only): {np.mean(full_times_nonzero):.2f}ms")
else:
    print("Avg full-path time (escalated only): n/a (no escalations)")

# visualization of routing decisions
fig, ax = plt.subplots(figsize=(6, 4))

routes = ['Cheap Path', 'Full Path']
counts = [cheap_routes, full_routes]
colors = ['#1abc9c', '#e67e22']

bars = ax.bar(routes, counts, color=colors, alpha=0.85, edgecolor='black', linewidth=1)
ax.set_ylabel('Requests')
ax.set_title('Adaptive Inference Routing')
ax.grid(axis='y', alpha=0.25)

for bar, value in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f'{value}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

Adaptive inference can reduce average compute in benign traffic, but should be stress-tested against confidence manipulation attacks.

## PART 4: Rate Limiting and Queue Management <a id="ratelimit"></a>

- Without rate limiting, queue depth should grow.
- With rate limiting, queue depth should remain bounded.
- Key metric to watch: rejected/throttled request percentage under overload.

In [ ]:
@dataclass
class RateLimiter:
    """Token bucket rate limiter."""
    max_qps: float = 100  # Queries per second
    burst_size: int = 10  # Allow burst of up to 10 requests
    tokens: float = 0.0
    last_refill: float = field(default_factory=time.time)
    
    def __post_init__(self):
        self.tokens = float(self.burst_size)

    def refill_tokens(self):
        """Add tokens based on elapsed time."""
        now = time.time()
        elapsed = now - self.last_refill
        tokens_to_add = elapsed * self.max_qps
        self.tokens = min(self.burst_size, self.tokens + tokens_to_add)
        self.last_refill = now

    def refill_tokens_at(self, now: float):
        """Add tokens using simulated time."""
        elapsed = now - self.last_refill
        tokens_to_add = elapsed * self.max_qps
        self.tokens = min(self.burst_size, self.tokens + tokens_to_add)
        self.last_refill = now
    
    def check_rate_limit(self, tokens_needed: int = 1) -> Tuple[bool, float]:
        """Check if request can proceed.
        
        Returns:
            (allowed, wait_time_ms)
        """
        self.refill_tokens()
        
        if self.tokens >= tokens_needed:
            self.tokens -= tokens_needed
            return True, 0.0
        else:
            wait_time = (tokens_needed - self.tokens) / self.max_qps
            return False, wait_time * 1000

    def check_rate_limit_at(self, now: float, tokens_needed: int = 1) -> Tuple[bool, float]:
        """Check limit using simulated arrival time."""
        self.refill_tokens_at(now)
        if self.tokens >= tokens_needed:
            self.tokens -= tokens_needed
            return True, 0.0
        wait_time = (tokens_needed - self.tokens) / self.max_qps
        return False, wait_time * 1000

@dataclass
class RequestQueue:
    """Queue management with priority."""
    max_queue_size: int = 100
    queue: deque = field(default_factory=deque)
    requests_rejected: int = field(default=0)
    
    def add_request(self, request_id: int, priority: int = 0) -> bool:
        """Add request to queue.
        
        Returns:
            Whether request was enqueued
        """
        if len(self.queue) >= self.max_queue_size:
            self.requests_rejected += 1
            return False
        
        self.queue.append((request_id, priority))
        return True
    
    def get_queue_depth(self) -> int:
        return len(self.queue)

print("\n[1] Simulating attack without rate limiting...")

# Simulate 500 normal requests + 500 attack requests arriving in 5 seconds
normal_arrival_rate = 100  # per second
attack_arrival_rate = 100  # per second (sponge attacks)
simulation_time = 5.0  # seconds

# Without rate limiting
queue_no_limit = RequestQueue(max_queue_size=10000)

normal_requests = int(normal_arrival_rate * simulation_time)
attack_requests = int(attack_arrival_rate * simulation_time)
total_requests = normal_requests + attack_requests
arrival_qps = normal_arrival_rate + attack_arrival_rate
service_qps = 100
arrival_times = np.arange(total_requests) / arrival_qps

queue_depth = 0.0
peak_queue_no_limit = 0.0
for _ in arrival_times:
    queue_depth += 1.0
    queue_depth = max(0.0, queue_depth - service_qps / arrival_qps)
    peak_queue_no_limit = max(peak_queue_no_limit, queue_depth)
pending_no_limit = int(round(queue_depth))
for i in range(pending_no_limit):
    queue_no_limit.add_request(i)

print(f"Requests arrived: {total_requests}")
print(f"Requests rejected: {queue_no_limit.requests_rejected}")
print(f"Service can process {100} QPS, but receives {normal_arrival_rate + attack_arrival_rate} QPS")
print(f"-> Queue builds up: {len(queue_no_limit.queue)} pending requests (peak~{peak_queue_no_limit:.1f})")

print("\n[2] With rate limiting (100 QPS token bucket)...")

rate_limiter = RateLimiter(max_qps=100, burst_size=10)
queue_with_limit = RequestQueue(max_queue_size=100)

allowed_requests = 0
rejected_requests = 0
total_wait_time = 0
queue_depth = 0.0
peak_queue_with_limit = 0.0

for i, t in enumerate(arrival_times):
    allowed, wait_ms = rate_limiter.check_rate_limit_at(now=float(t), tokens_needed=1)
    
    if allowed:
        allowed_requests += 1
        queue_depth += 1.0
    else:
        # Backoff
        total_wait_time += wait_ms
        rejected_requests += 1

    queue_depth = max(0.0, queue_depth - service_qps / arrival_qps)
    peak_queue_with_limit = max(peak_queue_with_limit, queue_depth)

pending_with_limit = int(round(queue_depth))
for i in range(pending_with_limit):
    queue_with_limit.add_request(i)

print(f"Requests allowed by rate limiter: {allowed_requests}")
print(f"Requests rejected (rate limited): {rejected_requests}")
print(f"Queue depth: {queue_with_limit.get_queue_depth()}")
print(f"Average wait time per rejected: {total_wait_time / max(rejected_requests, 1):.2f}ms")

print("\n[3] Impact:")
print(f"  Without limit: Queue grows uncontrollably ({len(queue_no_limit.queue)} requests, peak~{peak_queue_no_limit:.1f})")
print(f"  With limit: Queue stays bounded ({queue_with_limit.get_queue_depth()} requests, peak~{peak_queue_with_limit:.1f})")
print("  -> Rate limiting bounds ingress load but does not classify benign vs malicious requests")

# Visualization of queue depths
fig, ax = plt.subplots(figsize=(7, 4))

scenarios = ['No Defense', 'Rate Limited']
depths = [len(queue_no_limit.queue), queue_with_limit.get_queue_depth()]
colors = ['#c0392b', '#27ae60']

bars = ax.bar(scenarios, depths, color=colors, alpha=0.85, edgecolor='black', linewidth=1)
ax.set_ylabel('Pending Requests')
ax.set_title('Queue Depth Under Load')
ax.grid(axis='y', alpha=0.25)

for bar, depth in zip(bars, depths):
    ax.text(bar.get_x() + bar.get_width() / 2, depth, f'{int(depth)}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## PART 5: Anomaly Detection (Latency-Based) <a id="anomaly"></a>

- During the attack phase, anomaly flags should increase.
- In normal/recovery phases, anomaly rate should decrease.
- Key metrics to watch: true positive rate, false positive rate, detection latency.

In [ ]:
class LatencyAnomalyDetector:
    """Detect sponge attacks via inference latency anomalies."""
    
    def __init__(self, window_size: int = 100, threshold_std: float = 3.0):
        self.window_size = window_size
        self.threshold_std = threshold_std
        self.latencies = deque(maxlen=window_size)
        self.anomaly_count = 0
    
    def add_latency(self, latency_ms: float) -> Tuple[bool, float, float]:
        """Check if latency is anomalous.
        
        Returns:
            (is_anomaly, mean_latency, std_latency)
        """
        self.latencies.append(latency_ms)
        
        if len(self.latencies) < 10:
            return False, 0, 0  # Not enough data
        
        mean = np.mean(list(self.latencies))
        std = np.std(list(self.latencies))
        
        # Z-score test
        z_score = abs(latency_ms - mean) / max(std, 1e-6)
        is_anomaly = z_score > self.threshold_std
        
        if is_anomaly:
            self.anomaly_count += 1
        
        return is_anomaly, mean, std
    
    def get_anomaly_rate(self) -> float:
        """Fraction of anomalous requests."""
        return self.anomaly_count / len(self.latencies) if len(self.latencies) > 0 else 0

print("\n[1] Simulating request stream with anomaly detection...")

detector = LatencyAnomalyDetector(window_size=100, threshold_std=2.5)

# Simulate attack: sudden increase in high-latency requests around request 100
np.random.seed(42)
attack_latencies = np.concatenate([
    np.random.normal(10, 2, 100),      # 0-100: normal
    np.random.normal(50, 10, 100),     # 100-200: attack surge
    np.random.normal(10, 2, 50)        # 200-250: recovery
])

anomalies = []
means = []
stds = []

for lat in attack_latencies:
    is_anom, mean, std = detector.add_latency(lat)
    anomalies.append(is_anom)
    means.append(mean)
    stds.append(std)

print(f"Total requests: {len(attack_latencies)}")
print(f"Anomalies detected: {sum(anomalies)} ({100*sum(anomalies)/len(attack_latencies):.1f}%)")
print(f"False positives (normal phase): {sum(anomalies[:100])} out of 100")
print(f"True positives (attack phase): {sum(anomalies[100:200])} out of 100")
print(f"Recovery phase (post-attack): {sum(anomalies[200:])} out of 50")

print("\n[2] Detection Quality:")
true_positives = sum(anomalies[100:200])
false_positives = sum(anomalies[:100]) + sum(anomalies[200:])
print(f"  True positive rate: {100*true_positives/100:.1f}% (detect attacks)")
print(f"  False positive rate: {100*false_positives/150:.1f}% (false alarms)")
first_detection = next((idx for idx in range(100, 200) if anomalies[idx]), None)
if first_detection is None:
    print("  Detection latency: not detected during attack window")
else:
    print(f"  Detection latency: {first_detection - 100 + 1} requests into attack")

# Visualization of latency and anomalies
fig, ax = plt.subplots(figsize=(10, 4))

time_points = np.arange(len(attack_latencies))
colors = ['#e74c3c' if a else '#2ecc71' for a in anomalies]

ax.scatter(time_points, attack_latencies, c=colors, alpha=0.65, s=18)
ax.plot(time_points, means, color='#2c3e50', linewidth=2, label='Rolling Mean')
ax.axvspan(100, 200, alpha=0.15, color='red', label='Attack Window')

ax.set_xlabel('Request Number')
ax.set_ylabel('Latency (ms)')
ax.set_title('Latency-Based Anomaly Detection')
ax.legend(loc='upper left')
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## Summary: Sponge Attack Defenses

1. **Input validation** is cheap and blocks many obvious sponge inputs, but requires threshold tuning.
2. **Resource constraints** enforce practical safety budgets (latency, memory delta, batch size).
3. **Adaptive inference** can reduce average cost by routing easy inputs to cheaper paths.
4. **Rate limiting** is the strongest availability control during overload, independent of request intent.
5. **Anomaly detection** helps identify attack phases and trigger mitigation, with FP/TP trade-offs.

### Multi-Layer Defense Strategy

```
User Request
     ↓
Layer 1: Input Validation -> Reject malformed/suspicious inputs
     ↓ (pass)
Layer 2: Rate Limiting -> Control ingress under load
     ↓ (pass)
Layer 3: Inference Guard -> Enforce latency/memory/batch budgets
     ↓ (pass)
Layer 4: Anomaly Detection -> Detect shifts and trigger mitigation
     ↓
Response (if all checks pass)
```

**Overall Protection:** defense-in-depth significantly reduces attack success risk, but effectiveness depends on calibration, workload, and attacker adaptation.

---
## Exercises

### Exercise 1: Tuning Input Validation
Adjust the InputValidationPolicy thresholds to minimize false positives while maintaining detection rate:
- Try: max_height/width = 48×48, 64×64, 96×96
- Try: max_brightness_variance = 5, 10, 20
- Plot: Detection rate vs False positive rate
- Find optimal operating point

### Exercise 2: Adaptive Rate Limiting
Implement a rate limiter that adapts based on latency:
- If average latency > 50ms, reduce QPS limit by 20%
- If average latency < 10ms, increase QPS limit by 10%
- Can you maintain 100 QPS average while handling sponge attacks?

### Exercise 3: Defense Evasion
Design a sponge attack that evades the defenses:
- How small can the image perturbation be while still increasing latency 2×?
- Can you create a "stealthy" sponge that passes input validation?
- Can you spread the attack across multiple requests to evade rate limiting?

### Exercise 4: Cost-Benefit Analysis
Calculate the resource cost of each defense:
- Input validation: O(H×W) for Laplacian computation
- Rate limiting: O(1) token bucket
- Anomaly detection: O(window_size) for statistics

Which defense provides best protection per unit cost?

### Exercise 5: Sponge Detection via ML
Train a binary classifier to detect sponge inputs:
- Features: (height, width, brightness_var, edge_density, noise_level)
- Positive: Sponge inputs (high-res, noisy)
- Negative: Normal inputs
- Can an ML-based detector outperform hand-crafted rules?

### Exercise 6: Production Defense Pipeline
Design a complete defense system:

1. Input Validation (reject 60%)
2. Rate Limiting (reject 20% of remaining)
3. Timeout (kill 10% of remaining)
4. Anomaly Detection (alert on 80% of remaining)

What's the overall detection rate? Can you improve it?